In [9]:
from dotenv import load_dotenv
from tenacity import retry, wait_random_exponential, stop_after_attempt
import json
import os
from typing import Annotated, Dict

import autogen
from autogen import AssistantAgent, UserProxyAgent, register_function
from autogen.cache import Cache
from typing import Annotated
from typing import Annotated
from openai import AzureOpenAI
from typing import Annotated
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.identity import AzureCliCredential
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizedQuery
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    SearchIndex,
    AzureOpenAIVectorizer,
    AzureOpenAIParameters
)

load_dotenv()
# Configure environment variables
service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
index_name = os.getenv("AZURE_SEARCH_INDEX")
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_GPT4o_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4o_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
azure_openai_embedding_dimensions = 1536

In [10]:
# Configure OpenAI API
aoai_client = AzureOpenAI(
  azure_endpoint = AZURE_OPENAI_ENDPOINT, 
  api_key=AZURE_OPENAI_API_KEY,  
  api_version=AZURE_OPENAI_API_VERSION
)


llm_config = {
    "cache_seed": 43,  # change the cache_seed for different trials
    "temperature": 0,
    "timeout": 120,  # in seconds
    "config_list": 
    [
        {
            "model": AZURE_OPENAI_GPT4o_DEPLOYMENT_NAME,
            "api_type": "azure",
            "api_key": AZURE_OPENAI_API_KEY,
            "base_url": AZURE_OPENAI_ENDPOINT,
            "api_version": AZURE_OPENAI_API_VERSION
        }
    ]
}



In [11]:
from azure.search.documents.models import VectorizedQuery
credential = AzureKeyCredential(key)
search_client = SearchClient(endpoint=service_endpoint, index_name=index_name, credential=credential)

# Generate Document Embeddings using OpenAI Ada Model
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def calc_embeddings(text):
    # model = "deployment_name"
    embeddings = aoai_client.embeddings.create(input = [text], model=AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME).data[0].embedding
    return embeddings

def do_search(query: Annotated[str, "a query"])-> Annotated[str, "results"]:
    fields = "embedding"
    embedding = calc_embeddings(query)
    vector_query = VectorizedQuery(vector=embedding, k_nearest_neighbors=3, fields=fields)
  
    results = search_client.search(  
        search_text=None,  
        vector_queries= [vector_query],
        select=["content"],
    )  
    answer = ''
    for result in results:  
        print(f"Score: {result['@search.score']}")  
        print(f"Content: {result['content']}")  
        answer = answer + result['content']
    return answer

In [13]:
user_proxy = autogen.UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the WebSearchEngineer and AzureAISearcher and decides who to assign the task to.",
    code_execution_config={
        "work_dir": "code",
        "use_docker": False
    },
    human_input_mode="TERMINATE",
)

web_search_engineer = autogen.AssistantAgent(
    name="WebSearchEngineer",
    llm_config=llm_config,
    system_message="""Web Search Engineer. Make sure you save code to disk.  You write python/shell 
    code to solve tasks. Wrap the code in a code block that specifies the script type and the name of the file to 
    save to disk.""",
)

aia_searcher = autogen.AssistantAgent(
    name="AzureAISearcher",
    llm_config=llm_config,
    system_message="""
        Assistant who has extra content retrieval power for questions about Microsoft Fabric, 
        only use the functions you have been provided with. 
        Reply TERMINATE when the task is done.""",
)

# Register the tool signature with the assistant agent.
aia_searcher.register_for_llm(name="do_search", description="Get information about Microsoft Fabric.")(do_search)
# Register the tool function with the user proxy agent.
user_proxy.register_for_execution(name="do_search")(do_search)


group_chat = autogen.GroupChat(
    agents=[user_proxy, web_search_engineer, aia_searcher], messages=[], max_round=12
)
manager = autogen.GroupChatManager(groupchat=group_chat, llm_config=llm_config)

In [14]:
user_proxy.initiate_chat(
    manager,
    message="What is Microsoft Fabric?",
)

Admin (to chat_manager):

What is Microsoft Fabric?

--------------------------------------------------------------------------------

Next speaker: AzureAISearcher

AzureAISearcher (to chat_manager):

***** Suggested tool call (call_6Rb35x2C69Myveaimbwe99F6): do_search *****
Arguments: 
{"query":"What is Microsoft Fabric?"}
**************************************************************************

--------------------------------------------------------------------------------

Next speaker: Admin


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION do_search...
Score: 0.91130716
Content: ） Important
Microsoft Fabric is currently in PREVIEW. This information relates to a prerelease
product that may be substantially modified before it's released. Microsoft makes no
warranties, expressed or implied, with respect to the information provided here.
Score: 0.9052431
Content: ）  Important
Microsoft Fabric is currently in PREVIEW. This information relates to a prerelease
product that 

ChatResult(chat_id=None, chat_history=[{'content': 'What is Microsoft Fabric?', 'role': 'assistant', 'name': 'Admin'}, {'content': 'None', 'tool_calls': [{'id': 'call_6Rb35x2C69Myveaimbwe99F6', 'function': {'arguments': '{"query":"What is Microsoft Fabric?"}', 'name': 'do_search'}, 'type': 'function'}], 'name': 'AzureAISearcher', 'role': 'assistant'}, {'content': "） Important\nMicrosoft Fabric is currently in PREVIEW. This information relates to a prerelease\nproduct that may be substantially modified before it's released. Microsoft makes no\nwarranties, expressed or implied, with respect to the information provided here.）  Important\nMicrosoft Fabric is currently in PREVIEW. This information relates to a prerelease\nproduct that may be substantially modified before it's released. Microsoft makes no\nwarranties, expressed or implied, with respect to the information provided here.\nNext stepsLakehouse end-to-end scenario:\noverview and architecture\nArticle• 05/23/2023\nMicrosoft Fabric

In [15]:
user_proxy.initiate_chat(
    manager,
    message="What is the weather today in Miami?",
)

Admin (to chat_manager):

What is the weather today in Miami?

--------------------------------------------------------------------------------

Next speaker: WebSearchEngineer

WebSearchEngineer (to chat_manager):

I'm unable to browse the internet in real-time to provide current weather updates. However, you can check the latest weather in Miami by visiting a weather website like Weather.com, AccuWeather, or using a weather app on your smartphone.

--------------------------------------------------------------------------------

Next speaker: Admin


>>>>>>>> USING AUTO REPLY...
Admin (to chat_manager):



--------------------------------------------------------------------------------

Next speaker: Admin


>>>>>>>> USING AUTO REPLY...
Admin (to chat_manager):



--------------------------------------------------------------------------------

Next speaker: WebSearchEngineer

WebSearchEngineer (to chat_manager):

It looks like you didn't provide any additional information or ask a n

ChatResult(chat_id=None, chat_history=[{'content': 'What is the weather today in Miami?', 'role': 'assistant', 'name': 'Admin'}, {'content': "I'm unable to browse the internet in real-time to provide current weather updates. However, you can check the latest weather in Miami by visiting a weather website like Weather.com, AccuWeather, or using a weather app on your smartphone.", 'name': 'WebSearchEngineer', 'role': 'user'}, {'content': '', 'role': 'assistant', 'name': 'Admin'}, {'content': '', 'role': 'assistant', 'name': 'Admin'}, {'content': "It looks like you didn't provide any additional information or ask a new question. How can I assist you further?", 'name': 'WebSearchEngineer', 'role': 'user'}, {'content': '', 'role': 'assistant', 'name': 'Admin'}, {'content': '', 'role': 'assistant', 'name': 'Admin'}, {'content': 'It seems like there might be some confusion. If you need assistance with a specific task or have a question, please let me know how I can help!', 'name': 'WebSearchE